# Notebook 1 — Labels (CBERT)

At every release **A** that a later release was built on, take **every production file** from git. Each
one gets a label for the interval **A → B** that follows: **1** if it was refactored before release
**B**, **0** if not. The releases form a **tree** (`intervals_<repo>.json`): B's parent A is the latest
release whose Java code B contains, so an interval never mixes two branches. When several releases were
built on A (a fork), A's file is labelled 1 if it was refactored on any of them.

The rows are all production files that git lists at A (Notebook 0). No file is dropped for any other
reason: annotation types and every other kind of Java file are rows too.

Every file RefactoringMiner reported as refactored between A and B is accounted for, interval by
interval, with exactly one outcome, so nothing disappears without being counted.

### Four terms

| term | meaning |
|---|---|
| **Java commit** | any commit that touched a `.java` file |
| **refactoring commit** | a commit in which RefactoringMiner detected at least one refactoring |
| **refactoring operation** | one detected refactoring; a commit can hold many |
| **refactored file** | a file on the left side of at least one refactoring operation, the code as it was before the change. Counted once per interval |

**Move Source Folder** relocates a whole source tree. It reorganises the repository rather than
changing any class, so it is not counted as a refactoring; it is still used to follow files to
their new location.

**Writes** (to `outputs/notebook_1_cbert/`): `labels_<repo>.csv` (one row per production file per
labelled release), `refactored_files_<repo>.csv`, `interval_table_<repo>.csv`, `label_summary.csv`.
Notebook 2 reads `labels_<repo>.csv` and `label_summary.csv` from this folder.

Cell 0 downloads the mined inputs from Zenodo. Run Notebook 0 first: this notebook reads its
`production_scope.json` and `production_files_<repo>.csv`.

### Cell 0 — The mined inputs, from Zenodo
Notebooks 0–3 read what was mined from the five repos: the release grids, Quarry's outputs and a git
clone of each repo. They are published on Zenodo, [10.5281/zenodo.24182007](https://doi.org/10.5281/zenodo.24182007),
as one zip and one git bundle per repo.

For each repo this cell downloads only what is missing from `workspace_data/`:
- the zip, checked against the record's `SHA256SUMS.txt` and unpacked into `workspace_data/`, every file
  checked against the zip's `MANIFEST_<repo>.json`;
- the bundle, checked the same way and restored as the clone in `workspace_data/repos/<repo>/`.

A workspace that already holds the files and clones (for example one that ran Quarry itself) downloads
nothing. The same cell opens Notebooks 0, 1, 2 and 3, so any of them can be run first. The first run
downloads about 2.5 GB.

In [ ]:
import os, json, hashlib, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

# The workspace Quarry uses: QUARRY_HOME if set, otherwise workspace_data beside the codebase.
WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
MINED_RECORD = "24182007"                         # Zenodo record of the mined inputs
MINED_REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]
NEEDED = ["versions/rel_hist_{r}.json", "outputs/{r}/intervals_{r}.json", "outputs/{r}/refactorings_{r}.jsonl",
          "outputs/{r}/ledger_history_{r}.jsonl", "outputs/{r}/diffs_{r}.jsonl", "outputs/{r}/links_{r}.csv"]


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def have_mined_files(repo):
    """Every file the notebooks need, in outputs/<repo>/ or outputs/ (both layouts are read)."""
    def there(rel):
        p = WORKSPACE / rel.format(r=repo)
        return p.exists() or (WORKSPACE / "outputs" / p.name).exists()
    return all(there(x) for x in NEEDED)


def download(name, dest, sums):
    """One file of the record, checked against SHA256SUMS.txt before it is kept."""
    tmp = dest.with_name(dest.name + ".part")
    print(f"   downloading {name}")
    urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/{name}?download=1", tmp)
    if sha256_of(tmp) != sums[name]:
        tmp.unlink()
        raise SystemExit(f"{name}: SHA-256 differs from the record's SHA256SUMS.txt; download it again")
    tmp.replace(dest)


def fetch_mined(repos):
    downloads = WORKSPACE / "downloads"
    downloads.mkdir(parents=True, exist_ok=True)
    sums = None
    for repo in repos:
        clone = WORKSPACE / "repos" / repo
        need_files, need_clone = not have_mined_files(repo), not (clone / ".git").exists()
        if not (need_files or need_clone):
            print(f"{repo:<18}mined files and clone already in {WORKSPACE}")
            continue
        if sums is None:
            urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/SHA256SUMS.txt?download=1",
                                       downloads / "SHA256SUMS.txt")
            sums = {n: h for h, n in (l.split() for l in
                    (downloads / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines() if l.strip())}
        if need_files:
            z = downloads / f"{repo}_mined.zip"
            download(z.name, z, sums)
            with zipfile.ZipFile(z) as zf:
                manifest = json.loads(zf.read(f"MANIFEST_{repo}.json"))
                for f in manifest["files"]:
                    out = WORKSPACE / f["file"]
                    out.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(f["file"]) as src, open(out, "wb") as dst:
                        shutil.copyfileobj(src, dst, 1 << 24)
                    if sha256_of(out) != f["sha256"]:
                        raise SystemExit(f"{f['file']}: SHA-256 differs from MANIFEST_{repo}.json")
                (WORKSPACE / "versions" / f"MANIFEST_{repo}.json").write_bytes(zf.read(f"MANIFEST_{repo}.json"))
            z.unlink()
            print(f"{repo:<18}{len(manifest['files'])} mined files unpacked and checked")
        if need_clone:
            b = downloads / f"{repo}.bundle"
            download(b.name, b, sums)
            clone.mkdir(parents=True, exist_ok=True)
            for args in (["clone", "--mirror", "--quiet", str(b.resolve()), str(clone / ".git")],
                         ["-C", str(clone), "config", "--bool", "core.bare", "false"],
                         ["-C", str(clone), "-c", "core.longpaths=true", "reset", "--hard", "--quiet"]):
                r = subprocess.run(["git", *args], capture_output=True, text=True)
                if r.returncode != 0:
                    raise SystemExit(f"git {' '.join(args[:2])} failed for {repo}: {r.stderr.strip()[:300]}")
            b.unlink()
            print(f"{repo:<18}clone restored from {b.name}")


fetch_mined(MINED_REPOS)

In [1]:
import os, re, json, subprocess, collections
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 500)

WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
OUTPUTS, CLONES = WORKSPACE / "outputs", WORKSPACE / "repos"
OUT = OUTPUTS / "notebook_1_cbert"                  # everything this notebook writes goes here
OUT.mkdir(parents=True, exist_ok=True)
REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]

SCOPES = json.loads((OUTPUTS / "production_scope.json").read_text(encoding="utf-8"))["repositories"]


def find(repo, name):
    for d in (OUTPUTS / repo, OUTPUTS):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found in {OUTPUTS / repo} or {OUTPUTS}")


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def norm(path):
    return str(path).replace("\\", "/").lstrip("/")


def production_rule(scope):
    """Notebook 0's rule: the most specific known root is approved, and the file declares a type."""
    roots = sorted(scope["roots"], key=len, reverse=True)
    approved = set(scope["approved"])
    def is_production(path):
        root = next((r for r in roots if path.startswith(r)), None)
        return (root in approved and path.endswith(".java")
                and path.rsplit("/", 1)[-1] not in {"package-info.java", "module-info.java"})
    return is_production

## Step 1 — Intervals

The release tree, from `intervals_<repo>.json` (written by `build_release_tree.py --write-intervals`):
every release's parent is the latest release whose Java code it contains exactly, and every
interval is an **edge** parent → release holding `git rev-list parent..release`. Every commit belongs
to the first edge that contains it. Every commit belongs to the
first release that contains it. Refactoring commits that belong to no interval — before the first
release, or after the last — are counted here and play no further part.

In [2]:
def topological_order(repo, release_shas):
    """
    sha -> position in git's history, parents always before children.

    Moves are ordered by this, NOT by commit timestamps. Timestamps can run backwards -- rebased,
    imported or scripted commits keep dates out of step with history -- and a rename chain
    followed in date order then stops part-way. questdb's NFSdb rebrand did exactly that.
    """
    out = subprocess.run(["git", "rev-list", "--topo-order", "--reverse", *release_shas],
                         cwd=CLONES / repo, capture_output=True, text=True).stdout.split()
    return {sha: i for i, sha in enumerate(out)}


INTERVALS, INTERVAL_OF, RELEASE_POS, ORDER, TREE = {}, {}, {}, {}, {}
for repo in REPOS:
    d = json.loads(find(repo, f"intervals_{repo}.json").read_text(encoding="utf-8"))
    if d.get("kind") != "tree":
        raise SystemExit(f"{repo}: intervals_{repo}.json is the one-list version -- run "
                         f"build_release_tree.py --write-intervals first")
    ivs = d["interval_index"]                                      # the edges, parent -> release
    position = {iv["interval"]: i for i, iv in enumerate(ivs)}
    INTERVALS[repo], TREE[repo] = ivs, d["releases"]
    INTERVAL_OF[repo] = {sha: position[name] for sha, name in d["map"].items()}
    RELEASE_POS[repo] = {r["tag"]: r["pos"] for r in d["releases"]}  # release -> history-size order
    ORDER[repo] = topological_order(repo, sorted(r["sha"] for r in d["releases"]))

    inside = outside = 0
    for rec in read_jsonl(find(repo, f"refactorings_{repo}.jsonl")):
        if rec.get("refactorings"):
            if rec["sha1"] in INTERVAL_OF[repo]:
                inside += 1
            else:
                outside += 1
    forks = sum(len(r["children"]) > 1 for r in d["releases"])
    print(f"{repo:<18}{len(d['releases']):>4} releases   {len(ivs):>4} edges   "
          f"{len(d['labelled_releases']):>4} labelled releases   {forks:>3} forks   "
          f"refactoring commits: inside {inside:,}   outside every edge {outside:,}")

checkstyle         203 releases    202 edges    201 labelled releases     1 forks   refactoring commits: inside 3,792   outside every edge 36
commons-lang        35 releases     33 edges     28 labelled releases     4 forks   refactoring commits: inside 1,872   outside every edge 138
dubbo              123 releases    122 edges    106 labelled releases    15 forks   refactoring commits: inside 2,900   outside every edge 505
junit-framework     61 releases     60 edges     48 labelled releases    10 forks   refactoring commits: inside 1,967   outside every edge 1,567
questdb            119 releases    118 edges    114 labelled releases     4 forks   refactoring commits: inside 3,241   outside every edge 49


## Step 2 — Files at each release

The production files at each release, **from git** (Notebook 0, `production_files_<repo>.csv`). Every
one of them at a labelled release becomes a row in step 6.

In [3]:
AT, SEEN = {}, {}
for repo in REPOS:
    pf = pd.read_csv(find(repo, f"production_files_{repo}.csv"))
    pf = pf[pf.category == "production"]
    AT[repo] = {tag: set(g.path) for tag, g in pf.groupby("release")}

    seen = collections.defaultdict(set)                  # production path -> release positions
    for tag, paths in AT[repo].items():
        if tag in RELEASE_POS[repo]:
            for p in paths:
                seen[p].add(RELEASE_POS[repo][tag])
    SEEN[repo] = seen

    starts = sorted({iv["from"] for iv in INTERVALS[repo]})      # a fork point counts once
    missing = [t for t in starts if t not in AT[repo]]
    if missing:
        raise SystemExit(f"{repo}: no production files listed for {len(missing)} labelled release(s), "
                         f"e.g. {missing[:3]}")
    print(f"{repo:<18}production files at the {len(starts)} labelled releases: "
          f"{sum(len(AT[repo][t]) for t in starts):,}")

checkstyle        production files at the 201 labelled releases: 70,105
commons-lang      production files at the 28 labelled releases: 4,124
dubbo             production files at the 106 labelled releases: 140,453
junit-framework   production files at the 48 labelled releases: 28,545
questdb           production files at the 114 labelled releases: 218,893


## Step 3 — Refactored files in each edge

For each refactoring operation in a commit between A and B, the `.java` files on its **left
side**. All of them — tests included — so that every file RefactoringMiner reported is accounted
for in step 5.

In [4]:
NOT_A_REFACTORING = {"move source folder"}

REFACTORED, RM_RECORDS = {}, {}
for repo in REPOS:
    refactored = collections.defaultdict(set)              # interval -> refactored paths
    records = []                                           # (interval, order, refactorings)
    for rec in read_jsonl(find(repo, f"refactorings_{repo}.jsonl")):
        i = INTERVAL_OF[repo].get(rec["sha1"])
        if i is None or not rec.get("refactorings"):
            continue
        records.append((i, ORDER[repo].get(rec["sha1"], -1), rec["refactorings"]))
        for r in rec["refactorings"]:
            if str(r.get("type", "")).strip().lower() in NOT_A_REFACTORING:
                continue
            for loc in r.get("leftSideLocations") or []:
                path = norm(loc.get("filePath", ""))
                if path.endswith(".java"):
                    refactored[i].add(path)
    REFACTORED[repo], RM_RECORDS[repo] = refactored, records
    print(f"{repo:<18}{sum(len(v) for v in refactored.values()):>8,} refactored files "
          f"(counted once per edge)")

checkstyle          11,406 refactored files (counted once per edge)
commons-lang         2,446 refactored files (counted once per edge)
dubbo               11,594 refactored files (counted once per edge)
junit-framework      4,104 refactored files (counted once per edge)
questdb             20,094 refactored files (counted once per edge)


## Step 4 — Renames and moves

A refactored file may carry a different name from the one it had at A. Two records of renames:

- **RefactoringMiner** — Move Class, Rename Class and Move And Rename Class move one file; Move Source
  Folder and Rename Package move a folder. A Move Source Folder with no file paths still names its
  folders in its description.
- **git** — `git log -M` lists every file git considers renamed in each commit (by its default rule:
  at least 50% of the content unchanged).

Moves are put in order by their place in git's history, not by commit date: dates can run
backwards (rebased or imported commits), and a chain of renames followed by date stops part-way.

RefactoringMiner is consulted first. git counts only where RefactoringMiner has no record of the
move — and step 5 reports the two separately.

In [5]:
MOVE_TYPES = {"move class", "rename class", "move and rename class",
              "move source folder", "rename package"}
FOLDER_TYPES = {"move source folder", "rename package"}
FOLDER_TEXT = re.compile(r"Move Source Folder\s+(\S+)\s+to\s+(\S+)", re.I)


def java_paths(locations):
    return list(dict.fromkeys(norm(l.get("filePath", "")) for l in (locations or [])
                              if str(l.get("filePath", "")).endswith(".java")))


def file_moves(r):
    left, right = java_paths(r.get("leftSideLocations")), java_paths(r.get("rightSideLocations"))
    if len(left) == 1 and len(right) == 1:
        return [(left[0], right[0])] if left[0] != right[0] else []
    by_name = {p.rsplit("/", 1)[-1]: p for p in right}
    return [(p, by_name[p.rsplit("/", 1)[-1]]) for p in left
            if p.rsplit("/", 1)[-1] in by_name and by_name[p.rsplit("/", 1)[-1]] != p]


def folder_of(old, new):
    a, b = old.split("/"), new.split("/")
    k = 0
    while k < min(len(a), len(b)) - 1 and a[-1 - k] == b[-1 - k]:
        k += 1
    fa, fb = "/".join(a[:len(a) - k]), "/".join(b[:len(b) - k])
    return (fa, fb) if k and fa and fb and fa != fb else None


def refactoringminer_moves(repo):
    moves = []                                             # (interval, order, old, new, kind)
    for i, ts, refs in RM_RECORDS[repo]:
        for r in refs:
            t = str(r.get("type", "")).strip().lower()
            if t not in MOVE_TYPES:
                continue
            pairs = file_moves(r)
            moves += [(i, ts, o, n, "file") for o, n in pairs]
            if t in FOLDER_TYPES:
                folders = {folder_of(o, n) for o, n in pairs} - {None}
                m = FOLDER_TEXT.search(str(r.get("description", "")))
                if m:
                    folders.add((m.group(1).strip("/"), m.group(2).strip("/")))
                moves += [(i, ts, o, n, "folder") for o, n in folders]
    return moves


def git_renames(repo):
    """Renames git detects, commit by commit, in every interval."""
    moves = []
    for i, iv in enumerate(INTERVALS[repo]):
        out = subprocess.run(
            ["git", "-c", "core.quotepath=off", "log", "-M", "--diff-filter=R", "--name-status",
             "--format=@%H", f"{iv['from_sha']}..{iv['to_sha']}", "--", "*.java"],
            cwd=CLONES / repo, capture_output=True, text=True, encoding="utf-8",
            errors="replace").stdout
        ts = -1
        for line in out.splitlines():
            if line.startswith("@"):
                ts = ORDER[repo].get(line[1:], -1)       # place in history, not commit date
            elif line.startswith("R"):
                parts = line.split("\t")
                if len(parts) == 3 and parts[1].endswith(".java") and parts[2].endswith(".java"):
                    moves.append((i, ts, norm(parts[1]), norm(parts[2]), "file"))
    return moves


class MoveIndex:
    """Moves grouped by interval, for following a path forwards or backwards through them."""

    def __init__(self, moves):
        self.fwd = collections.defaultdict(lambda: collections.defaultdict(list))
        self.bwd = collections.defaultdict(lambda: collections.defaultdict(list))
        self.folders = collections.defaultdict(list)
        for i, ts, o, n, kind in moves:
            if kind == "file":
                self.fwd[i][o].append((ts, n))
                self.bwd[i][n].append((ts, o))
            else:
                self.folders[i].append((ts, o, n))
        for d in (self.fwd, self.bwd):
            for per in d.values():
                for v in per.values():
                    v.sort()
        for v in self.folders.values():
            v.sort()
        self.intervals = sorted(set(self.fwd) | set(self.folders))

    def back(self, path, i):
        """Undo interval i's moves, newest first: the name the file had when i began."""
        t = float("inf")
        for _ in range(100):
            c = [(ts, o) for ts, o in self.bwd[i].get(path, ()) if ts < t]
            c += [(ts, o + path[len(n):]) for ts, o, n in self.folders[i]
                  if ts < t and path.startswith(n + "/")]
            if not c:
                return path
            t, path = max(c)
        return path

    def forward(self, path, earlier):
        """Replay the moves of the edges that led to this edge's start, oldest first."""
        for j in earlier:
            t = float("-inf")
            for _ in range(100):
                c = [(ts, n) for ts, n in self.fwd[j].get(path, ()) if ts > t]
                c += [(ts, n + path[len(o):]) for ts, o, n in self.folders[j]
                      if ts > t and path.startswith(o + "/")]
                if not c:
                    break
                t, path = min(c)
        return path


RM_INDEX, ALL_INDEX = {}, {}
for repo in REPOS:
    rm, git = refactoringminer_moves(repo), git_renames(repo)
    RM_INDEX[repo] = MoveIndex(rm)
    ALL_INDEX[repo] = MoveIndex(rm + git)
    print(f"{repo:<18}RefactoringMiner: {sum(m[4] == 'file' for m in rm):,} file moves, "
          f"{sum(m[4] == 'folder' for m in rm):,} folder moves   git: {len(git):,} renames")

checkstyle        RefactoringMiner: 5,373 file moves, 201 folder moves   git: 4,912 renames
commons-lang      RefactoringMiner: 117 file moves, 14 folder moves   git: 924 renames
dubbo             RefactoringMiner: 3,332 file moves, 262 folder moves   git: 4,859 renames
junit-framework   RefactoringMiner: 637 file moves, 77 folder moves   git: 860 renames
questdb           RefactoringMiner: 8,075 file moves, 80 folder moves   git: 6,606 renames


## Step 5 — Map each refactored file to release A

Each refactored file gets exactly **one** outcome, tried in this order:

| # | outcome | meaning |
|---|---|---|
| 1 | **not production** | test, example or documentation code — or under no root seen at any release |
| 2 | **same path** | the file is at A under the same name |
| 3 | **moved (RefactoringMiner)** | renamed or moved after A, traced back through RefactoringMiner's moves |
| 4 | **moved (git)** | traced back only once git's renames are added |
| 5 | **created during the interval** | did not exist at A; it first appears at a release with a larger history |
| 6 | **never in a release** | is at no release at all: it existed only between releases |
| 7 | **gone before A** | existed at a release with a smaller history, but not at A |

A file is traced two ways: this edge's moves undone, newest first (it moved, then was refactored),
and the moves of the edges that led to A replayed, oldest first (the refactoring was made on a branch
older than a move). Only A's own ancestry is replayed: moves made on another branch never touched A.

In [6]:
OUTCOMES = ["not production", "same path", "moved (RefactoringMiner)", "moved (git)",
            "created during the interval", "never in a release", "gone before A"]
STRENGTH = {"same path": 0, "moved (RefactoringMiner)": 1, "moved (git)": 2}


def every_earlier_name(index, path, i, limit=5000):
    """Every name the file could have had when interval i began, trying all moves, fewest moves first."""
    names, seen, todo = [], set(), collections.deque([(path, float("inf"))])
    while todo and len(seen) < limit:
        p, t = todo.popleft()
        if (p, t) in seen:
            continue
        seen.add((p, t))
        names.append(p)
        todo += [(o, ts) for ts, o in index.bwd[i].get(p, ()) if ts < t]
        todo += [(o + p[len(n):], ts) for ts, o, n in index.folders[i]
                 if ts < t and p.startswith(n + "/")]
    return names
 
 
def trace(index, path, i, at_a, earlier):
    """The file's path at A: undo this edge's moves, replay A's ancestry's, else try every chain."""
    for p in (index.back(path, i), index.forward(path, earlier)):
        if p in at_a:
            return p
    return next((p for p in every_earlier_name(index, path, i) if p in at_a), None)


MAPPED = {}
for repo in REPOS:
    is_production = production_rule(SCOPES[repo])
    rows = []
    for i, iv in enumerate(INTERVALS[repo]):
        at_a = AT[repo].get(iv["from"], set())
        earlier, a_pos = iv["ancestor_edges"], RELEASE_POS[repo][iv["from"]]
        for path in sorted(REFACTORED[repo].get(i, ())):
            target = None
            if not is_production(path):
                outcome = "not production"
            elif path in at_a:
                outcome, target = "same path", path
            elif (target := trace(RM_INDEX[repo], path, i, at_a, earlier)) is not None:
                outcome = "moved (RefactoringMiner)"
            elif (target := trace(ALL_INDEX[repo], path, i, at_a, earlier)) is not None:
                outcome = "moved (git)"
            else:
                seen = SEEN[repo].get(path, set())
                outcome = ("never in a release" if not seen
                           else "created during the interval" if min(seen) > a_pos
                           else "gone before A")
            rows.append((i, iv["interval"], path, outcome, target or ""))
    MAPPED[repo] = M = pd.DataFrame(rows, columns=["pos", "interval", "refactored_file",
                                                   "outcome", "path_at_A"])
    M.to_csv(OUT / f"refactored_files_{repo}.csv", index=False)
    c = M.outcome.value_counts()
    print(f"{repo:<18}" + "   ".join(f"{o} {int(c.get(o, 0)):,}" for o in OUTCOMES))

checkstyle        not production 7,566   same path 3,630   moved (RefactoringMiner) 40   moved (git) 1   created during the interval 135   never in a release 34   gone before A 0
commons-lang      not production 1,300   same path 792   moved (RefactoringMiner) 130   moved (git) 17   created during the interval 116   never in a release 91   gone before A 0
dubbo             not production 4,304   same path 6,022   moved (RefactoringMiner) 316   moved (git) 24   created during the interval 621   never in a release 268   gone before A 39
junit-framework   not production 1,971   same path 1,780   moved (RefactoringMiner) 30   moved (git) 8   created during the interval 240   never in a release 75   gone before A 0
questdb           not production 4,448   same path 11,492   moved (RefactoringMiner) 971   moved (git) 82   created during the interval 945   never in a release 2,156   gone before A 0


## Step 6 — Labels

Every production file at A gets **1** if at least one refactored file, on any edge that starts at A,
mapped to it, and **0** otherwise. A fork point therefore gets one row per file, labelled from all the
branches built on it. A release with no edge starting at it (the end of a maintenance line, and the
newest release) has no finished interval and no rows. Two refactored files can map to one file at A
(`Old.java`, and `New.java` after the rename), and that is one label.

In [7]:
LABELS, TARGETS = {}, {}
for repo in REPOS:
    M = MAPPED[repo]
    targets, edges_of = {}, collections.defaultdict(list)
    for i, iv in enumerate(INTERVALS[repo]):                     # per edge: path at A -> how reached
        hits = M[(M.pos == i) & (M.path_at_A != "")]
        best = {}
        for p, o in zip(hits.path_at_A, hits.outcome):
            if p not in best or STRENGTH[o] < STRENGTH[best[p]]:
                best[p] = o
        targets[i] = best
        edges_of[iv["from"]].append(i)

    parts = []
    for rel in sorted(edges_of, key=RELEASE_POS[repo].get):      # per release: all its edges at once
        best = {}
        for i in edges_of[rel]:
            for p, o in targets[i].items():
                if p not in best or STRENGTH[o] < STRENGTH[best[p]]:
                    best[p] = o
        paths = sorted(AT[repo][rel])
        parts.append(pd.DataFrame({
            "interval": " + ".join(INTERVALS[repo][i]["interval"] for i in edges_of[rel]),
            "pos": RELEASE_POS[repo][rel], "release": rel, "path": paths,
            "refactored": [int(p in best) for p in paths],
            "reached_by": [best.get(p, "") for p in paths]}))
    TARGETS[repo] = targets
    LABELS[repo] = L = pd.concat(parts, ignore_index=True)
    L.insert(0, "repo", repo)
    if L.duplicated(["release", "path"]).any():
        raise SystemExit(f"{repo}: a (release, file) appears twice")
    L.to_csv(OUT / f"labels_{repo}.csv", index=False)
    forks = sum(len(v) > 1 for v in edges_of.values())
    print(f"{repo:<18}{len(L):>8,} labelled file-releases at {len(edges_of)} releases ({forks} forks)   "
          f"{int(L.refactored.sum()):>7,} refactored ({L.refactored.mean():.1%} of all file-releases)")

checkstyle          70,105 labelled file-releases at 201 releases (1 forks)     3,640 refactored (5.2% of all file-releases)
commons-lang         4,124 labelled file-releases at 28 releases (4 forks)       793 refactored (19.2% of all file-releases)
dubbo              140,453 labelled file-releases at 106 releases (15 forks)     5,805 refactored (4.1% of all file-releases)
junit-framework     28,545 labelled file-releases at 48 releases (10 forks)     1,590 refactored (5.6% of all file-releases)
questdb            218,893 labelled file-releases at 114 releases (4 forks)    11,365 refactored (5.2% of all file-releases)


## Step 7 — The per-edge table

One row per edge. In every row:

- the seven outcome columns add up to **refactored**
- **labelled 1** is no larger than **files at A**

The notebook stops if either fails. Totals are in the last row; the full tables are in
`interval_table_<repo>.csv`. A fork point's files appear in the row of each of its edges, so the
summary counts files, rows and labels per release instead, once each.

In [8]:
SHORT = {"not production": "not prod", "same path": "same path",
         "moved (RefactoringMiner)": "moved RM", "moved (git)": "moved git",
         "created during the interval": "created", "never in a release": "never",
         "gone before A": "gone before"}
TABLES, SUMMARY = {}, []
for repo in REPOS:
    M = MAPPED[repo]
    counts = M.groupby(["pos", "outcome"]).size().unstack(fill_value=0) \
              .reindex(columns=OUTCOMES, fill_value=0)
    rows = []
    for i, iv in enumerate(INTERVALS[repo]):
        at_a = AT[repo].get(iv["from"], set())
        c = counts.loc[i] if i in counts.index else pd.Series(0, index=OUTCOMES)
        mapped = M[(M.pos == i) & (M.path_at_A != "")]
        row = {"interval": iv["interval"], "files at A": len(at_a),
               "refactored": len(REFACTORED[repo].get(i, ())),
               **{SHORT[o]: int(c[o]) for o in OUTCOMES},
               "labelled 1": len(set(TARGETS[repo][i]) & at_a)}
        if sum(row[SHORT[o]] for o in OUTCOMES) != row["refactored"]:
            raise SystemExit(f"{repo} {iv['interval']}: outcomes do not add up to refactored")
        if row["labelled 1"] > row["files at A"]:
            raise SystemExit(f"{repo} {iv['interval']}: more labels than files at A")
        rows.append(row)
    T = pd.DataFrame(rows)
    total = T.drop(columns=["interval"]).sum()
    total["interval"] = "TOTAL"
    T = pd.concat([T, total.to_frame().T], ignore_index=True)
    TABLES[repo] = T
    T.to_csv(OUT / f"interval_table_{repo}.csv", index=False)

    print(f"\n{'=' * 120}\n{repo}   ({len(T) - 1} edges; every row adds up. A fork point's files appear in "
          f"each of its edges, so the TOTAL row counts them once per edge)\n{'=' * 120}")
    print(T.to_string(index=False))
    t = T.iloc[-1]
    prod = int(t["refactored"]) - int(t["not prod"])
    mapped = int(t["same path"] + t["moved RM"] + t["moved git"])
    L = LABELS[repo]
    SUMMARY.append({"repo": repo, "edges": len(T) - 1, "labelled releases": L.release.nunique(),
                    "refactored files": int(t["refactored"]), "production": prod,
                    "mapped to A": mapped,
                    "mapped share of production": round(mapped / max(1, prod), 3),
                    "labelled 1": int(L.refactored.sum()),
                    "labelled rows": len(L),
                    "rate": round(L.refactored.mean(), 4)})

S = pd.DataFrame(SUMMARY)
S.to_csv(OUT / "label_summary.csv", index=False)
print(f"\n\n{'=' * 120}\nSUMMARY\n{'=' * 120}")
print(S.to_string(index=False))
print(f"\nwrote interval_table_<repo>.csv, refactored_files_<repo>.csv, labels_<repo>.csv, "
      f"label_summary.csv to {OUT}")


checkstyle   (202 edges; every row adds up. A fork point's files appear in each of its edges, so the TOTAL row counts them once per edge)
                                interval files at A refactored not prod same path moved RM moved git created never gone before labelled 1
                release1_1 -> release1_2         13          5        1         3        0         0       1     0           0          3
                release1_2 -> release1_3         14          6        2         4        0         0       0     0           0          4
                release1_3 -> release1_4         21         11        1         9        0         0       1     0           0          9
                release1_4 -> release2_0         22          1        0         1        0         0       0     0           0          1
                release2_0 -> release2_2         22         15        2        12        0         0       1     0           0         12
                release2_2 -> rel